# Milestone 1 — Dataset and deterministic evaluation foundation

## Goal

Build a trustworthy, reproducible, and leakage-safe measurement foundation before introducing any language model.

At this checkpoint we have completed the **dataset-freezing portion** of Milestone 1. We have not yet built the result-normalization and SQL-result comparison logic.

### Mental model

A model benchmark is an exam:

- FINCH supplies questions and an intended SQL answer key.
- `working_dev` is the visible practice set used to develop the system.
- `frozen_eval` is the sealed final exam used for fair comparisons.
- The answer key must be audited before model performance can be trusted.

This notebook records how the slice was selected, how reference-data problems were discovered, how narrowly scoped compatibility rules were validated, and how the final evaluation manifest was frozen.


## 0. Environment

This notebook expects the local FINCH metadata at `../data/finch_dataset.json` and the BIRD `financial.sqlite` database at `../data/financial.sqlite`. Both files stay outside Git because the repository stores reproducible setup logic and metadata rather than redistributing the dataset.


## 1. Inventory the FINCH financial slice

FINCH contains 75,725 natural-language/SQL pairs across several financial databases. We narrow the first experiment to the BIRD `financial` database so the schema and failure modes remain understandable.

The inventory establishes the available population before we select anything:

- 106 examples belong to BIRD `financial`.
- All 106 carry FINCH's upstream `dev` label, so DomainAdapt must define its own internal roles.
- Difficulty distribution: 62 easy, 37 medium, and 7 hard.
- IDs 1–3 were viewed during project orientation and are excluded from later selection.


In [17]:
from collections import Counter
from pathlib import Path
import json

dataset_path = Path("../data/finch_dataset.json")
records = json.loads(dataset_path.read_text(encoding="utf-8"))

financial = [
    row for row in records
    if row["db_id"] == "financial" and row["db_name"] == "bird"
]

ids = [row["question_id"] for row in financial]

print("Total FINCH records:", len(records))
print("BIRD financial records:", len(financial))
print("Partition counts:", dict(Counter(row["partition"] for row in financial)))
print("Difficulty counts:", dict(Counter(row["difficulty"] for row in financial)))
print("Question ID range:", min(ids), "to", max(ids))
print("Unique question IDs:", len(set(ids)))
print("Known orientation IDs present:", sorted(set(ids) & {1, 2, 3}))

Total FINCH records: 75725
BIRD financial records: 106
Partition counts: {'dev': 106}
Difficulty counts: {'medium': 37, 'easy': 62, 'hard': 7}
Question ID range: 1 to 106
Unique question IDs: 106
Known orientation IDs present: [1, 2, 3]


## 2. Create reproducible working and frozen sets

Think of the benchmark as an exam:

| Internal role | Size | Easy | Medium | Hard | Use |
|---|---:|---:|---:|---:|---|
| `working_dev` | 12 | 6 | 4 | 2 | Visible practice set for evaluator, prompt, and harness development |
| `frozen_eval` | 24 | 12 | 9 | 3 | Hidden exam used consistently across model and system variants |

Selection is deterministic rather than hand-picked. Each eligible question receives a SHA-256 rank derived from a fixed salt, its difficulty, and its ID. Anyone using the same files and rules will reproduce the same candidates.

The file hashes act as fingerprints for the exact FINCH JSON and SQLite database used. If an upstream file changes, validation will fail instead of silently changing the benchmark.

### Leakage boundary

- `working_dev` may be inspected, but it cannot be used as training or few-shot data.
- `frozen_eval` questions, SQL, and results remain unopened during development.
- All 106 examples from this database are excluded from future SFT, synthetic-training prompts, and few-shot pools. Later training examples must come from other databases.


In [18]:
from collections import Counter, defaultdict
from pathlib import Path
import hashlib
import json
import sqlite3

dataset_path = Path("../data/finch_dataset.json").resolve()
db_path = Path("../data/financial.sqlite").resolve()

records = json.loads(dataset_path.read_text(encoding="utf-8"))
financial = [
    row for row in records
    if row["db_id"] == "financial" and row["db_name"] == "bird"
]

ORIENTATION_IDS = {1, 2, 3}
SELECTION_SALT = "domain-adapt-financial-v1"

split_spec = {
    "working_dev": {"easy": 6, "medium": 4, "hard": 2},
    "frozen_eval": {"easy": 12, "medium": 9, "hard": 3},
}

def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def selection_rank(row: dict) -> str:
    value = (
        f"{SELECTION_SALT}:"
        f"{row['difficulty']}:"
        f"{row['question_id']}"
    )
    return hashlib.sha256(value.encode("utf-8")).hexdigest()

eligible = [
    row for row in financial
    if row["question_id"] not in ORIENTATION_IDS
]

by_difficulty = defaultdict(list)
for row in eligible:
    by_difficulty[row["difficulty"]].append(row)

for difficulty in by_difficulty:
    by_difficulty[difficulty].sort(key=selection_rank)

selected = {"working_dev": [], "frozen_eval": []}
offsets = Counter()

for role, difficulty_counts in split_spec.items():
    for difficulty, count in difficulty_counts.items():
        start = offsets[difficulty]
        stop = start + count
        selected[role].extend(by_difficulty[difficulty][start:stop])
        offsets[difficulty] = stop

print("Dataset SHA-256:", file_sha256(dataset_path))
print("Database SHA-256:", file_sha256(db_path))

for role, rows in selected.items():
    print(f"\n{role}:")
    print("  total:", len(rows))
    print("  difficulty:", dict(Counter(r["difficulty"] for r in rows)))
    print("  IDs:", sorted(r["question_id"] for r in rows))

Dataset SHA-256: c1e462743e4891fecf0fc10dbc6b0eb554fac56a186bdeedec9ccd662e0a4130
Database SHA-256: d15d89cdb068a202b6f2b99342af44dffc1d52545b39ceaf62efdc0ba570101e

working_dev:
  total: 12
  difficulty: {'easy': 6, 'medium': 4, 'hard': 2}
  IDs: [5, 11, 14, 16, 45, 49, 61, 69, 81, 89, 99, 105]

frozen_eval:
  total: 24
  difficulty: {'easy': 12, 'medium': 9, 'hard': 3}
  IDs: [4, 6, 10, 12, 13, 17, 28, 29, 30, 35, 37, 38, 39, 41, 47, 52, 53, 58, 62, 80, 82, 93, 96, 104]


## 3. Audit the supplied answer key before trusting it

FINCH supplies reference SQL—the benchmark's intended answer key. Successful execution alone does not prove that a query is meaningful: an incorrect filter can still execute and return an empty result.

This first audit flags mechanically suspicious outcomes:

- execution errors;
- no result rows;
- any `NULL` value;
- a single numeric zero.

These signals are not automatically errors. They tell us which examples require investigation before entering a trusted benchmark.


In [19]:
def audit_reference(row: dict, conn: sqlite3.Connection) -> dict:
    try:
        cursor = conn.execute(row["SQL"])
        result = cursor.fetchall()
        column_count = len(cursor.description or [])

        concerns = []

        if len(result) == 0:
            concerns.append("empty result")

        if any(value is None for result_row in result for value in result_row):
            concerns.append("contains NULL")

        if (
            len(result) == 1
            and len(result[0]) == 1
            and isinstance(result[0][0], (int, float))
            and result[0][0] == 0
        ):
            concerns.append("single numeric zero")

        return {
            "question_id": row["question_id"],
            "difficulty": row["difficulty"],
            "executes": True,
            "row_count": len(result),
            "column_count": column_count,
            "concerns": concerns,
        }

    except sqlite3.Error as error:
        return {
            "question_id": row["question_id"],
            "difficulty": row["difficulty"],
            "executes": False,
            "row_count": None,
            "column_count": None,
            "concerns": [f"{type(error).__name__}: {error}"],
        }

conn = sqlite3.connect(f"file:{db_path.as_posix()}?mode=ro", uri=True)

for role, rows in selected.items():
    audits = [audit_reference(row, conn) for row in rows]
    flagged = [
        audit
        for audit in audits
        if not audit["executes"] or audit["concerns"]
    ]

    print(f"\n{role}:")
    print("  executed:", sum(a["executes"] for a in audits), "/", len(audits))
    print("  flagged:", flagged)


working_dev:
  executed: 12 / 12
  flagged: [{'question_id': 99, 'difficulty': 'easy', 'executes': True, 'row_count': 1, 'column_count': 1, 'concerns': ['single numeric zero']}, {'question_id': 14, 'difficulty': 'easy', 'executes': True, 'row_count': 0, 'column_count': 1, 'concerns': ['empty result']}, {'question_id': 49, 'difficulty': 'medium', 'executes': True, 'row_count': 1, 'column_count': 1, 'concerns': ['single numeric zero']}, {'question_id': 5, 'difficulty': 'medium', 'executes': True, 'row_count': 1, 'column_count': 1, 'concerns': ['single numeric zero']}, {'question_id': 11, 'difficulty': 'medium', 'executes': True, 'row_count': 0, 'column_count': 1, 'concerns': ['empty result']}, {'question_id': 105, 'difficulty': 'medium', 'executes': True, 'row_count': 0, 'column_count': 3, 'concerns': ['empty result']}, {'question_id': 81, 'difficulty': 'hard', 'executes': True, 'row_count': 1, 'column_count': 1, 'concerns': ['contains NULL']}]

frozen_eval:
  executed: 24 / 24
  flagge

## 4. Diagnose suspicious examples using only `working_dev`

We may inspect the visible working set without contaminating the hidden evaluation set. The following cell prints only flagged working examples so we can compare the question, reference SQL, and observed result.

The initial audit found seven suspicious working queries. Their pattern suggested a systematic incompatibility between FINCH's SQL and the supplied SQLite database rather than seven unrelated semantic mistakes.


In [20]:
working_audits = {
    row["question_id"]: audit_reference(row, conn)
    for row in selected["working_dev"]
}

flagged_working = [
    row
    for row in selected["working_dev"]
    if working_audits[row["question_id"]]["concerns"]
]

for row in sorted(flagged_working, key=lambda item: item["question_id"]):
    audit = working_audits[row["question_id"]]
    result_preview = conn.execute(row["SQL"]).fetchmany(10)

    print("=" * 80)
    print("Question ID:", row["question_id"])
    print("Difficulty:", row["difficulty"])
    print("Question:", row["question"])
    print("Reference SQL:", row["SQL"])
    print("Result preview:", result_preview)
    print("Concern:", audit["concerns"])

Question ID: 5
Difficulty: medium
Question: How many male customers who are living in North Bohemia have average salary greater than 8000?
Reference SQL: SELECT COUNT(t1.client_id) FROM client AS t1 INNER JOIN district AS t2 ON t1.district_id = t2.district_id WHERE t1.gender = 'm' AND t2.a3 = 'north bohemia' AND t2.a11 > 8000
Result preview: [(0,)]
Concern: ['single numeric zero']
Question ID: 11
Difficulty: medium
Question: Among the accounts who have loan validity more than 12 months, list out the accounts that have the highest approved amount and have account opening date in 1993.
Reference SQL: SELECT t1.account_id FROM loan AS t1 INNER JOIN account AS t2 ON t1.account_id = t2.account_id WHERE STRFTIME('%y', t2.date) = '1993' AND t1.duration > 12 ORDER BY t1.amount DESC LIMIT 1
Result preview: []
Concern: ['empty result']
Question ID: 14
Difficulty: easy
Question: State different accounts who have account opening date before 1997 and own an amount of money greater than 3000USD
Refe

## 5. Confirm the database representations

The diagnostic queries revealed two benchmark compatibility problems:

1. **Categorical value casing.** FINCH reference SQL uses literals such as `'m'`, `'owner'`, and `'c'`, while the database stores `M`, `OWNER`, and `C`. SQLite's default equality comparison treats those as different values.
2. **Invalid SQLite year formatting.** FINCH uses `STRFTIME('%y', date)`, but SQLite does not support `%y`. It returns `NULL`. The database stores ISO-style dates, and `%Y` correctly returns their four-digit year.

These are answer-key/database mismatches. They must be resolved before any model is scored.


In [21]:
diagnostics = {
    "client.gender": """
        SELECT gender, COUNT(*)
        FROM client
        GROUP BY gender
        ORDER BY gender
    """,
    "loan.status": """
        SELECT status, COUNT(*)
        FROM loan
        GROUP BY status
        ORDER BY status
    """,
    "disp.type": """
        SELECT type, COUNT(*)
        FROM disp
        GROUP BY type
        ORDER BY type
    """,
    "account.frequency": """
        SELECT frequency, COUNT(*)
        FROM account
        GROUP BY frequency
        ORDER BY frequency
    """,
    "North Bohemia spelling": """
        SELECT DISTINCT A3
        FROM district
        WHERE LOWER(A3) = 'north bohemia'
    """,
    "account date behavior": """
        SELECT
            date,
            TYPEOF(date),
            STRFTIME('%Y', date),
            STRFTIME('%y', date)
        FROM account
        LIMIT 5
    """,
    "loan date behavior": """
        SELECT
            date,
            TYPEOF(date),
            STRFTIME('%Y', date),
            STRFTIME('%y', date)
        FROM loan
        LIMIT 5
    """,
}

for label, sql in diagnostics.items():
    print(f"\n{label}:")
    print(conn.execute(sql).fetchall())


client.gender:
[('F', 2645), ('M', 2724)]

loan.status:
[('A', 203), ('B', 31), ('C', 403), ('D', 45)]

disp.type:
[('DISPONENT', 869), ('OWNER', 4500)]

account.frequency:
[('POPLATEK MESICNE', 4167), ('POPLATEK PO OBRATU', 93), ('POPLATEK TYDNE', 240)]

North Bohemia spelling:
[('north Bohemia',)]

account date behavior:
[('1995-03-24', 'text', '1995', None), ('1993-02-26', 'text', '1993', None), ('1997-07-07', 'text', '1997', None), ('1996-02-21', 'text', '1996', None), ('1997-05-30', 'text', '1997', None)]

loan date behavior:
[('1994-01-05', 'text', '1994', None), ('1996-04-29', 'text', '1996', None), ('1997-12-08', 'text', '1997', None), ('1998-10-14', 'text', '1998', None), ('1998-04-19', 'text', '1998', None)]


## 6. Define explicit reference compatibility rules

We introduce two narrow, documented transformations:

- replace unsupported `STRFTIME('%y', ...)` with `STRFTIME('%Y', ...)`;
- apply `COLLATE NOCASE` to reference-SQL equality comparisons against string literals.

The transformation applies **only to FINCH reference SQL** when producing expected results. Model-generated SQL will later execute exactly as generated against the untouched database. The evaluator therefore does not repair model mistakes.

This regex implementation is exploratory notebook logic. Once the behavior stabilizes, it can be replaced by a more robust reusable implementation.


In [22]:
import re

def make_reference_compatible(sql: str) -> tuple[str, list[str]]:
    compatible = sql
    applied_rules = []

    # FINCH uses a four-digit year literal but SQLite does not support %y.
    updated = re.sub(
        r"STRFTIME\(\s*'%y'\s*,",
        "STRFTIME('%Y',",
        compatible,
        flags=re.IGNORECASE,
    )
    if updated != compatible:
        applied_rules.append("sqlite_four_digit_year")
        compatible = updated

    # FINCH lowercases categorical literals while the database retains casing.
    updated = re.sub(
        r"\b([A-Za-z_][A-Za-z0-9_]*(?:\.[A-Za-z_][A-Za-z0-9_]*)?)"
        r"\s*=\s*('(?:''|[^'])*')",
        r"\1 = \2 COLLATE NOCASE",
        compatible,
    )
    if updated != compatible:
        applied_rules.append("case_insensitive_text_equality")
        compatible = updated

    return compatible, applied_rules

### Validate the repairs on flagged working examples

Re-running the seven visible failures gives meaningful results after the compatibility rules. Examples changed from zero or `NULL` to substantive results, confirming that the rules address systematic dataset incompatibilities.


In [23]:
for row in sorted(flagged_working, key=lambda item: item["question_id"]):
    compatible_sql, applied_rules = make_reference_compatible(row["SQL"])
    result = conn.execute(compatible_sql).fetchmany(10)

    print("=" * 80)
    print("Question ID:", row["question_id"])
    print("Applied rules:", applied_rules)
    print("Compatible result:", result)

Question ID: 5
Applied rules: ['case_insensitive_text_equality']
Compatible result: [(280,)]
Question ID: 11
Applied rules: ['sqlite_four_digit_year']
Compatible result: [(10451,)]
Question ID: 14
Applied rules: ['sqlite_four_digit_year']
Compatible result: [(1,), (2,), (4,), (6,), (7,), (8,), (9,), (10,), (11,), (14,)]
Question ID: 49
Applied rules: ['case_insensitive_text_equality']
Compatible result: [(47,)]
Question ID: 81
Applied rules: ['sqlite_four_digit_year', 'case_insensitive_text_equality']
Compatible result: [(25.300191222790616,)]
Question ID: 99
Applied rules: ['case_insensitive_text_equality']
Compatible result: [(240,)]
Question ID: 105
Applied rules: ['case_insensitive_text_equality']
Compatible result: [(1, 18, 'Pisek'), (2, 1, 'Hl.m. Praha'), (4, 5, 'Kolin'), (6, 12, 'Pribram'), (7, 15, 'Cesky Krumlov'), (8, 51, 'Trutnov'), (9, 60, 'Prostejov'), (10, 57, 'Hodonin'), (12, 70, 'Karvina'), (13, 54, 'Brno - mesto')]


### Re-audit the complete working set

The compatible audit records which transformations were applied and repeats the same suspicious-result checks. All 12 working references must pass before the fixed rules are allowed to touch hidden candidates.


In [24]:
def audit_compatible_reference(
    row: dict,
    conn: sqlite3.Connection,
) -> dict:
    compatible_sql, applied_rules = make_reference_compatible(row["SQL"])

    try:
        cursor = conn.execute(compatible_sql)
        result = cursor.fetchall()
        column_count = len(cursor.description or [])

        concerns = []

        if len(result) == 0:
            concerns.append("empty result")

        if any(value is None for result_row in result for value in result_row):
            concerns.append("contains NULL")

        if (
            len(result) == 1
            and len(result[0]) == 1
            and isinstance(result[0][0], (int, float))
            and result[0][0] == 0
        ):
            concerns.append("single numeric zero")

        return {
            "question_id": row["question_id"],
            "difficulty": row["difficulty"],
            "executes": True,
            "row_count": len(result),
            "column_count": column_count,
            "applied_rules": applied_rules,
            "concerns": concerns,
        }

    except sqlite3.Error as error:
        return {
            "question_id": row["question_id"],
            "difficulty": row["difficulty"],
            "executes": False,
            "row_count": None,
            "column_count": None,
            "applied_rules": applied_rules,
            "concerns": [f"{type(error).__name__}: {error}"],
        }

working_compatible_audits = [
    audit_compatible_reference(row, conn)
    for row in selected["working_dev"]
]

print(
    "Executed:",
    sum(audit["executes"] for audit in working_compatible_audits),
    "/",
    len(working_compatible_audits),
)

print(
    "Still flagged:",
    [
        audit
        for audit in working_compatible_audits
        if not audit["executes"] or audit["concerns"]
    ],
)

Executed: 12 / 12
Still flagged: []


## 7. Audit frozen candidates without opening them

The rules were learned entirely from `working_dev`. We now apply them mechanically to frozen candidates while printing only IDs and audit metadata—never question text, SQL, or result values.

Two candidates remained ambiguous after compatibility repair: IDs 37 and 104. We do not inspect them because doing so would expose part of the hidden exam.


In [25]:
frozen_compatible_audits = [
    audit_compatible_reference(row, conn)
    for row in selected["frozen_eval"]
]

frozen_flagged = [
    audit
    for audit in frozen_compatible_audits
    if not audit["executes"] or audit["concerns"]
]

print(
    "Executed:",
    sum(audit["executes"] for audit in frozen_compatible_audits),
    "/",
    len(frozen_compatible_audits),
)
print("Still flagged:", frozen_flagged)

print(
    "Compatibility-rule usage:",
    dict(
        Counter(
            rule
            for audit in frozen_compatible_audits
            for rule in audit["applied_rules"]
        )
    ),
)

Executed: 24 / 24
Still flagged: [{'question_id': 104, 'difficulty': 'medium', 'executes': True, 'row_count': 1, 'column_count': 1, 'applied_rules': ['case_insensitive_text_equality'], 'concerns': ['contains NULL']}, {'question_id': 37, 'difficulty': 'hard', 'executes': True, 'row_count': 45, 'column_count': 1, 'applied_rules': ['case_insensitive_text_equality'], 'concerns': ['contains NULL']}]
Compatibility-rule usage: {'sqlite_four_digit_year': 6, 'case_insensitive_text_equality': 21}


### Check the wider eligible pool

Across all 103 non-orientation examples, every compatible reference query executes and only three remain mechanically ambiguous. This shows that the two compatibility rules explain nearly all of the observed mismatch rather than merely fitting the selected working examples.


In [26]:
all_compatible_audits = [
    audit_compatible_reference(row, conn)
    for row in eligible
]

all_flagged = [
    audit
    for audit in all_compatible_audits
    if not audit["executes"] or audit["concerns"]
]

print(
    "All eligible executed:",
    sum(audit["executes"] for audit in all_compatible_audits),
    "/",
    len(all_compatible_audits),
)
print("All eligible still flagged:", len(all_flagged))

print(
    "Remaining concern types:",
    dict(
        Counter(
            concern
            for audit in all_flagged
            for concern in audit["concerns"]
        )
    ),
)

print(
    "Compatibility-rule usage across eligible pool:",
    dict(
        Counter(
            rule
            for audit in all_compatible_audits
            for rule in audit["applied_rules"]
        )
    ),
)

All eligible executed: 103 / 103
All eligible still flagged: 3
Remaining concern types: {'contains NULL': 2, 'empty result': 1}
Compatibility-rule usage across eligible pool: {'case_insensitive_text_equality': 75, 'sqlite_four_digit_year': 24}


## 8. Replace ambiguous candidates conservatively

For this small V1 benchmark, a frozen reference must:

- execute successfully;
- return at least one row;
- contain no `NULL`;
- avoid a singleton numeric zero.

This conservative policy reduces the chance that an incorrect model query receives credit because both it and the reference accidentally return nothing.

IDs 37 and 104 are removed without inspection. Deterministic ranking selects clean replacements of the same difficulty: IDs 31 and 85.


In [27]:
audit_by_id = {
    audit["question_id"]: audit
    for audit in all_compatible_audits
}

def is_clean_reference(question_id: int) -> bool:
    audit = audit_by_id[question_id]
    return audit["executes"] and not audit["concerns"]

working_ids = {
    row["question_id"]
    for row in selected["working_dev"]
}

replacement_frozen = []

for difficulty, required_count in split_spec["frozen_eval"].items():
    candidates = [
        row
        for row in by_difficulty[difficulty]
        if row["question_id"] not in working_ids
        and is_clean_reference(row["question_id"])
    ]

    replacement_frozen.extend(candidates[:required_count])

old_frozen_ids = {
    row["question_id"]
    for row in selected["frozen_eval"]
}
new_frozen_ids = {
    row["question_id"]
    for row in replacement_frozen
}

print("Removed:", sorted(old_frozen_ids - new_frozen_ids))
print("Added:", sorted(new_frozen_ids - old_frozen_ids))
print("Final frozen IDs:", sorted(new_frozen_ids))
print(
    "Final difficulty counts:",
    dict(Counter(row["difficulty"] for row in replacement_frozen)),
)

selected["frozen_eval"] = replacement_frozen

Removed: [37, 104]
Added: [31, 85]
Final frozen IDs: [4, 6, 10, 12, 13, 17, 28, 29, 30, 31, 35, 38, 39, 41, 47, 52, 53, 58, 62, 80, 82, 85, 93, 96]
Final difficulty counts: {'easy': 12, 'medium': 9, 'hard': 3}


### Validate the final frozen set

The final set must contain exactly 24 clean references with the intended 12 easy, 9 medium, and 3 hard distribution. Assertions turn these benchmark assumptions into executable checks.


In [28]:
final_frozen_audits = [
    audit_compatible_reference(row, conn)
    for row in selected["frozen_eval"]
]

print("Total:", len(final_frozen_audits))
print(
    "Executed:",
    sum(audit["executes"] for audit in final_frozen_audits),
)
print(
    "Flagged:",
    [
        audit
        for audit in final_frozen_audits
        if not audit["executes"] or audit["concerns"]
    ],
)

assert len(final_frozen_audits) == 24
assert all(audit["executes"] for audit in final_frozen_audits)
assert all(not audit["concerns"] for audit in final_frozen_audits)

Total: 24
Executed: 24
Flagged: []


## 9. Freeze the benchmark manifest

The manifest is the sealed inventory for `domain-adapt-financial-v1`. It stores:

- exact source-file hashes;
- database and upstream partition identity;
- deterministic selection rules and salt;
- working and frozen IDs;
- orientation and ambiguous-reference exclusions;
- reference compatibility rules;
- audit requirements;
- leakage policy.

It records IDs and provenance rather than copying FINCH questions or SQL into the repository.


In [29]:
manifest = {
    "manifest_version": 1,
    "benchmark": "domain-adapt-financial-v1",
    "created_on": "2026-09-25",
    "source": {
        "dataset_repository": "domyn/FINCH",
        "dataset_file": "finch_dataset.json",
        "dataset_sha256": file_sha256(dataset_path),
        "database_file": "text2sql-db/text2sql/bird/financial.sqlite",
        "database_sha256": file_sha256(db_path),
        "license": "CC-BY-NC-4.0",
    },
    "scope": {
        "db_name": "bird",
        "db_id": "financial",
        "upstream_partition": "dev",
    },
    "selection": {
        "algorithm": (
            "SHA-256 rank within difficulty using "
            "'{salt}:{difficulty}:{question_id}'"
        ),
        "salt": SELECTION_SALT,
        "orientation_exclusions": sorted(ORIENTATION_IDS),
        "ambiguous_reference_exclusions": sorted(
            audit["question_id"]
            for audit in all_flagged
        ),
        "working_dev": {
            "ids": sorted(
                row["question_id"]
                for row in selected["working_dev"]
            ),
            "difficulty_counts": dict(
                Counter(
                    row["difficulty"]
                    for row in selected["working_dev"]
                )
            ),
        },
        "frozen_eval": {
            "ids": sorted(
                row["question_id"]
                for row in selected["frozen_eval"]
            ),
            "difficulty_counts": dict(
                Counter(
                    row["difficulty"]
                    for row in selected["frozen_eval"]
                )
            ),
        },
    },
    "reference_compatibility": {
        "scope": "reference_sql_only",
        "rules": [
            {
                "id": "sqlite_four_digit_year",
                "description": (
                    "Replace unsupported SQLite STRFTIME('%y', ...) "
                    "with STRFTIME('%Y', ...)."
                ),
            },
            {
                "id": "case_insensitive_text_equality",
                "description": (
                    "Apply COLLATE NOCASE to reference-SQL equality "
                    "comparisons against string literals because FINCH "
                    "lowercases literals while the database retains casing."
                ),
            },
        ],
    },
    "reference_audit": {
        "requirements": [
            "SQL executes successfully",
            "result contains at least one row",
            "result contains no NULL values",
            "result is not a singleton numeric zero",
        ],
    },
    "leakage_policy": {
        "working_dev": (
            "May be inspected for evaluator, prompt, and harness development; "
            "must not be used as few-shot or training data."
        ),
        "frozen_eval": (
            "Questions, SQL, and results must remain uninspected during "
            "system development and must never be used for prompting, "
            "few-shot selection, synthetic generation, or training."
        ),
        "financial_database": (
            "All 106 FINCH BIRD financial examples are excluded from "
            "few-shot pools and model-training data."
        ),
    },
}

manifest_path = Path("../configs/eval_manifest_v1.json")
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(
    json.dumps(manifest, indent=2) + "\n",
    encoding="utf-8",
)

print(manifest_path.resolve())
print(manifest_path.read_text(encoding="utf-8"))

C:\Users\shchitt\Downloads\Projects\domain-adapt\configs\eval_manifest_v1.json
{
  "manifest_version": 1,
  "benchmark": "domain-adapt-financial-v1",
  "created_on": "2026-09-25",
  "source": {
    "dataset_repository": "domyn/FINCH",
    "dataset_file": "finch_dataset.json",
    "dataset_sha256": "c1e462743e4891fecf0fc10dbc6b0eb554fac56a186bdeedec9ccd662e0a4130",
    "database_file": "text2sql-db/text2sql/bird/financial.sqlite",
    "database_sha256": "d15d89cdb068a202b6f2b99342af44dffc1d52545b39ceaf62efdc0ba570101e",
    "license": "CC-BY-NC-4.0"
  },
  "scope": {
    "db_name": "bird",
    "db_id": "financial",
    "upstream_partition": "dev"
  },
  "selection": {
    "algorithm": "SHA-256 rank within difficulty using '{salt}:{difficulty}:{question_id}'",
    "salt": "domain-adapt-financial-v1",
    "orientation_exclusions": [
      1,
      2,
      3
    ],
    "ambiguous_reference_exclusions": [
      37,
      43,
      104
    ],
    "working_dev": {
      "ids": [
        5,
 

### Validate the saved manifest

The final assertions verify source fingerprints, split sizes, and disjointness. If any of these change, the benchmark is no longer the frozen V1 evaluation set.


In [30]:
loaded_manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

assert loaded_manifest["source"]["dataset_sha256"] == (
    "c1e462743e4891fecf0fc10dbc6b0eb554fac56a186bdeedec9ccd662e0a4130"
)
assert loaded_manifest["source"]["database_sha256"] == (
    "d15d89cdb068a202b6f2b99342af44dffc1d52545b39ceaf62efdc0ba570101e"
)
assert len(loaded_manifest["selection"]["working_dev"]["ids"]) == 12
assert len(loaded_manifest["selection"]["frozen_eval"]["ids"]) == 24
assert set(
    loaded_manifest["selection"]["working_dev"]["ids"]
).isdisjoint(
    loaded_manifest["selection"]["frozen_eval"]["ids"]
)

print("Frozen manifest validated.")

Frozen manifest validated.


## Checkpoint summary

We started with 106 FINCH examples for one understandable banking database and produced:

- a visible 12-question `working_dev` set;
- a hidden 24-question `frozen_eval` set;
- deterministic selection and exact file provenance;
- database-level protection against training and few-shot leakage;
- two evidence-backed reference compatibility rules;
- a conservative audit that excludes ambiguous reference results;
- a committed manifest that reproduces the benchmark.

The SQLite database remains unchanged. Compatibility repair applies only to FINCH's reference SQL. Model-generated SQL will receive no repair.


## Deterministic result comparison

Correct SQL can be written in many different ways. Comparing SQL text would penalize a query that is syntactically different but returns the correct answer.

The evaluator therefore compares executed results:

- column count must match;
- duplicate rows are preserved;
- row order matters only when the reference SQL contains `ORDER BY`;
- small floating-point representation differences are tolerated;
- strings and `NULL` values remain semantically distinct;
- SQL execution failures are reported as failures rather than crashing the benchmark.

This is the first exploratory implementation. Its behavior will be tested in the notebook before extracting reusable code.

In [31]:
from collections import Counter
from dataclasses import dataclass
import math
import re


FLOAT_DECIMAL_PLACES = 6


@dataclass(frozen=True)
class QueryResult:
    rows: tuple[tuple[object, ...], ...]
    column_count: int


@dataclass(frozen=True)
class ComparisonResult:
    equivalent: bool
    reason: str
    expected_row_count: int | None
    candidate_row_count: int | None


def normalize_value(value: object) -> object:
    """Convert SQLite values into stable comparison values."""
    if isinstance(value, float):
        if math.isnan(value):
            return ("float", "nan")
        if math.isinf(value):
            return ("float", "inf" if value > 0 else "-inf")

        rounded = round(value, FLOAT_DECIMAL_PLACES)

        # Treat -0.0 and 0.0 as the same value.
        return 0.0 if rounded == 0 else rounded

    return value


def execute_sql(
    conn: sqlite3.Connection,
    sql: str,
) -> QueryResult:
    cursor = conn.execute(sql)
    rows = cursor.fetchall()

    normalized_rows = tuple(
        tuple(normalize_value(value) for value in row)
        for row in rows
    )

    return QueryResult(
        rows=normalized_rows,
        column_count=len(cursor.description or []),
    )


def reference_requires_order(sql: str) -> bool:
    """Initial ORDER BY detector; replace with parsed SQL later if needed."""
    return bool(
        re.search(r"\bORDER\s+BY\b", sql, flags=re.IGNORECASE)
    )


def compare_sql_results(
    conn: sqlite3.Connection,
    reference_sql: str,
    candidate_sql: str,
) -> ComparisonResult:
    try:
        expected = execute_sql(conn, reference_sql)
    except sqlite3.Error as error:
        return ComparisonResult(
            equivalent=False,
            reason=f"reference execution failed: {error}",
            expected_row_count=None,
            candidate_row_count=None,
        )

    try:
        candidate = execute_sql(conn, candidate_sql)
    except sqlite3.Error as error:
        return ComparisonResult(
            equivalent=False,
            reason=f"candidate execution failed: {error}",
            expected_row_count=len(expected.rows),
            candidate_row_count=None,
        )

    if expected.column_count != candidate.column_count:
        return ComparisonResult(
            equivalent=False,
            reason=(
                f"column count differs: "
                f"{expected.column_count} != {candidate.column_count}"
            ),
            expected_row_count=len(expected.rows),
            candidate_row_count=len(candidate.rows),
        )

    if reference_requires_order(reference_sql):
        equivalent = expected.rows == candidate.rows
        comparison_mode = "ordered"
    else:
        equivalent = Counter(expected.rows) == Counter(candidate.rows)
        comparison_mode = "unordered"

    return ComparisonResult(
        equivalent=equivalent,
        reason=(
            f"{comparison_mode} results match"
            if equivalent
            else f"{comparison_mode} results differ"
        ),
        expected_row_count=len(expected.rows),
        candidate_row_count=len(candidate.rows),
    )

In [32]:
comparison_checks = [
    {
        "name": "unordered row order is ignored",
        "reference": "SELECT 1 UNION ALL SELECT 2",
        "candidate": "SELECT 2 UNION ALL SELECT 1",
        "expected": True,
    },
    {
        "name": "ORDER BY makes row order meaningful",
        "reference": """
            SELECT 1 AS value
            UNION ALL
            SELECT 2
            ORDER BY value ASC
        """,
        "candidate": """
            SELECT 1 AS value
            UNION ALL
            SELECT 2
            ORDER BY value DESC
        """,
        "expected": False,
    },
    {
        "name": "duplicates are preserved",
        "reference": "SELECT 1 UNION ALL SELECT 1",
        "candidate": "SELECT 1",
        "expected": False,
    },
    {
        "name": "small float differences are tolerated",
        "reference": "SELECT 1.0 / 3.0",
        "candidate": "SELECT 0.3333334",
        "expected": True,
    },
    {
        "name": "different answers fail",
        "reference": "SELECT 1",
        "candidate": "SELECT 2",
        "expected": False,
    },
    {
        "name": "invalid candidate SQL fails cleanly",
        "reference": "SELECT 1",
        "candidate": "SELECT missing_column",
        "expected": False,
    },
]

for check in comparison_checks:
    result = compare_sql_results(
        conn,
        check["reference"],
        check["candidate"],
    )

    print(
        check["name"],
        "=>",
        result.equivalent,
        "|",
        result.reason,
    )

    assert result.equivalent is check["expected"]

print("All controlled comparison checks passed.")

unordered row order is ignored => True | unordered results match
ORDER BY makes row order meaningful => False | ordered results differ
duplicates are preserved => False | unordered results differ
small float differences are tolerated => True | unordered results match
different answers fail => False | unordered results differ
invalid candidate SQL fails cleanly => False | candidate execution failed: no such column: missing_column
All controlled comparison checks passed.


In [33]:
working_self_checks = []

for row in selected["working_dev"]:
    compatible_sql, _ = make_reference_compatible(row["SQL"])

    result = compare_sql_results(
        conn,
        reference_sql=compatible_sql,
        candidate_sql=compatible_sql,
    )

    working_self_checks.append(
        {
            "question_id": row["question_id"],
            "passed": result.equivalent,
            "reason": result.reason,
        }
    )

failed_self_checks = [
    check
    for check in working_self_checks
    if not check["passed"]
]

print(
    "Working references passed:",
    sum(check["passed"] for check in working_self_checks),
    "/",
    len(working_self_checks),
)
print("Failures:", failed_self_checks)

assert not failed_self_checks

Working references passed: 12 / 12
Failures: []


### Detect meaningful ordering through parsed SQL

A text search cannot tell whether `ORDER BY` controls the final returned rows or appears inside a nested query. We use SQLGlot to parse SQLite SQL and inspect the top-level query node.

This keeps the evaluator focused on result semantics:

- top-level `ORDER BY`: returned row order matters;
- nested `ORDER BY` only: final row order does not matter.

In [34]:
import sqlglot
from sqlglot.errors import ParseError


def reference_requires_order(sql: str) -> bool:
    """Return whether ORDER BY controls the final query result."""
    try:
        expression = sqlglot.parse_one(sql, read="sqlite")
    except ParseError as error:
        raise ValueError(
            f"Unable to determine ordering because SQL parsing failed: {error}"
        ) from error

    return expression.args.get("order") is not None

In [35]:
top_level_order = """
SELECT account_id
FROM account
ORDER BY account_id
"""

nested_order_only = """
SELECT COUNT(*)
FROM (
    SELECT account_id
    FROM account
    ORDER BY account_id
)
"""

print(
    "Top-level ORDER BY requires order:",
    reference_requires_order(top_level_order),
)
print(
    "Nested ORDER BY alone requires order:",
    reference_requires_order(nested_order_only),
)

assert reference_requires_order(top_level_order) is True
assert reference_requires_order(nested_order_only) is False

Top-level ORDER BY requires order: True
Nested ORDER BY alone requires order: False


In [36]:
ordering_audit = []
parse_failures = []

for row in eligible:
    compatible_sql, _ = make_reference_compatible(row["SQL"])

    regex_decision = bool(
        re.search(
            r"\bORDER\s+BY\b",
            compatible_sql,
            flags=re.IGNORECASE,
        )
    )

    try:
        parsed_decision = reference_requires_order(compatible_sql)
    except ValueError as error:
        parse_failures.append(
            {
                "question_id": row["question_id"],
                "error": str(error),
            }
        )
        continue

    if regex_decision != parsed_decision:
        ordering_audit.append(
            {
                "question_id": row["question_id"],
                "regex": regex_decision,
                "parsed": parsed_decision,
            }
        )

print("FINCH queries parsed:", len(eligible) - len(parse_failures), "/", len(eligible))
print("Parse failures:", parse_failures)
print("Regex/parser differences:", ordering_audit)

FINCH queries parsed: 103 / 103
Parse failures: []
Regex/parser differences: [{'question_id': 7, 'regex': True, 'parsed': False}, {'question_id': 50, 'regex': True, 'parsed': False}]


## Final Milestone 1 validation

The final gate exercises the evaluator through the committed manifest rather than relying on the notebook's temporary selection variables.

We validate three layers:

1. controlled examples prove the intended comparison behavior;
2. all 12 visible working references match themselves;
3. all 24 hidden frozen references match themselves without exposing their contents.

Passing this gate proves that the frozen reference set can be loaded, repaired according to documented compatibility rules, executed, normalized, and compared deterministically.

In [37]:
record_by_id = {
    row["question_id"]: row
    for row in financial
}


def run_reference_self_checks(
    role: str,
    question_ids: list[int],
) -> list[dict]:
    checks = []

    for question_id in question_ids:
        row = record_by_id[question_id]
        compatible_sql, applied_rules = make_reference_compatible(
            row["SQL"]
        )

        comparison = compare_sql_results(
            conn,
            reference_sql=compatible_sql,
            candidate_sql=compatible_sql,
        )

        checks.append(
            {
                "role": role,
                "question_id": question_id,
                "difficulty": row["difficulty"],
                "passed": comparison.equivalent,
                "reason": comparison.reason,
                "compatibility_rules": applied_rules,
            }
        )

    return checks


# Re-run the controlled behavioral checks with the parser-based
# ordering implementation.
controlled_failures = []

for check in comparison_checks:
    result = compare_sql_results(
        conn,
        check["reference"],
        check["candidate"],
    )

    if result.equivalent is not check["expected"]:
        controlled_failures.append(
            {
                "name": check["name"],
                "expected": check["expected"],
                "actual": result.equivalent,
                "reason": result.reason,
            }
        )


working_ids_from_manifest = loaded_manifest[
    "selection"
]["working_dev"]["ids"]

frozen_ids_from_manifest = loaded_manifest[
    "selection"
]["frozen_eval"]["ids"]

working_checks = run_reference_self_checks(
    "working_dev",
    working_ids_from_manifest,
)

frozen_checks = run_reference_self_checks(
    "frozen_eval",
    frozen_ids_from_manifest,
)

working_failures = [
    check for check in working_checks
    if not check["passed"]
]

frozen_failures = [
    check for check in frozen_checks
    if not check["passed"]
]

print(
    "Controlled checks:",
    len(comparison_checks) - len(controlled_failures),
    "/",
    len(comparison_checks),
)
print(
    "Working references:",
    len(working_checks) - len(working_failures),
    "/",
    len(working_checks),
)
print(
    "Frozen references:",
    len(frozen_checks) - len(frozen_failures),
    "/",
    len(frozen_checks),
)

print("Controlled failures:", controlled_failures)
print("Working failures:", working_failures)
print("Frozen failures:", frozen_failures)

assert not controlled_failures
assert not working_failures
assert not frozen_failures

print("Milestone 1 deterministic evaluation gate passed.")

Controlled checks: 6 / 6
Working references: 12 / 12
Frozen references: 24 / 24
Controlled failures: []
Working failures: []
Frozen failures: []
Milestone 1 deterministic evaluation gate passed.


## Validate the reusable implementation

The evaluator and FINCH compatibility behavior have now stabilized enough to move from exploratory notebook code into `src/domain_adapt/`.

This final integration check loads the reusable functions and runs the manifest-defined working and frozen sets through them. Passing proves that the extracted implementation preserves the notebook behavior.

In [42]:
from domain_adapt.evaluation import (
    compare_sql_results as compare_with_module,
)
from domain_adapt.finch import (
    make_reference_compatible as make_compatible_with_module,
)

ModuleNotFoundError: No module named 'domain_adapt'

In [41]:
from domain_adapt.evaluation import (
    compare_sql_results as compare_with_module,
)
from domain_adapt.finch import (
    make_reference_compatible as make_compatible_with_module,
)


module_validation = {}

for role in ("working_dev", "frozen_eval"):
    question_ids = loaded_manifest["selection"][role]["ids"]
    failures = []

    for question_id in question_ids:
        row = record_by_id[question_id]
        compatible_sql, _ = make_compatible_with_module(
            row["SQL"]
        )

        comparison = compare_with_module(
            conn,
            reference_sql=compatible_sql,
            candidate_sql=compatible_sql,
        )

        if not comparison.equivalent:
            failures.append(
                {
                    "question_id": question_id,
                    "reason": comparison.reason,
                }
            )

    module_validation[role] = {
        "total": len(question_ids),
        "passed": len(question_ids) - len(failures),
        "failures": failures,
    }

print(module_validation)

assert module_validation["working_dev"]["passed"] == 12
assert module_validation["frozen_eval"]["passed"] == 24
assert not module_validation["working_dev"]["failures"]
assert not module_validation["frozen_eval"]["failures"]

print("Reusable Milestone 1 evaluator validated.")

ModuleNotFoundError: No module named 'domain_adapt'